# Day 053 — Exercise 2: The JSON Contract

**What you'll build:** `post_chat(client, message, temperature)` — the frontend call that sends a chat message to the backend and reads the reply, honouring the shared JSON contract `{message, temperature}` → `{reply, model}`.

**Why it matters:** Frontend and backend agree on *shapes*: the keys the client sends must match what the API's `ChatRequest` expects, and the client reads back exactly the keys `ChatResponse` promises. Get a key wrong and you get a 422. `post_chat` also turns any non-200 into a plain error dict, so the UI always has something to show.

## Provided: Setup + Backend + check_health

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from starlette.testclient import TestClient
import httpx
import ollama


# ---- The AI backend (built on Day 52 — provided here) ----
class ChatRequest(BaseModel):
    message: str = Field(min_length=1, description='User message for the model')
    temperature: float = Field(default=0.7, ge=0.0, le=1.0)


class ChatResponse(BaseModel):
    reply: str
    model: str


class HealthResponse(BaseModel):
    status: str
    model: str


PROMPT_TEMPLATES = {
    'summary':  'Summarize the following topic in two sentences: {topic}',
    'explain':  'Explain {topic} to a complete beginner.',
    'critique': 'List three criticisms of {topic}.',
}


def run_model(model: str, prompt: str, temperature: float = 0.7) -> str:
    resp = ollama.chat(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': temperature},
    )
    return resp['message']['content'].strip()


def build_api(model: str = 'llama3.2') -> FastAPI:
    app = FastAPI(title='AI API', version='1.0.0')

    @app.get('/health', response_model=HealthResponse)
    def health():
        return HealthResponse(status='ok', model=model)

    @app.get('/templates')
    def list_templates():
        return {'templates': list(PROMPT_TEMPLATES.keys())}

    @app.post('/chat', response_model=ChatResponse)
    def chat(req: ChatRequest):
        try:
            return ChatResponse(reply=run_model(model, req.message, req.temperature),
                                model=model)
        except Exception as e:
            raise HTTPException(status_code=503, detail=f'Model unavailable: {e}')

    @app.post('/render/{name}', response_model=ChatResponse)
    def render_chat(name: str, req: ChatRequest):
        if name not in PROMPT_TEMPLATES:
            raise HTTPException(status_code=404, detail=f'template {name!r} not found')
        prompt = PROMPT_TEMPLATES[name].format(topic=req.message)
        try:
            return ChatResponse(reply=run_model(model, prompt, req.temperature),
                                model=model)
        except Exception as e:
            raise HTTPException(status_code=503, detail=f'Model unavailable: {e}')

    return app


def check_health(client) -> bool:
    """Ping the backend's GET /health through an injected HTTP client.

    Returns True only if the request succeeds with 200 AND status == 'ok'.
    Any exception (backend down, connection refused) -> False, never raises.
    The `client` is duck-typed: an httpx.Client in production, a TestClient in
    tests — both expose .get / .post / .request.
    """
    try:
        resp = client.get('/health')
        return resp.status_code == 200 and resp.json().get('status') == 'ok'
    except Exception:
        return False

## Your Implementation

In [ ]:
def post_chat(client, message: str, temperature: float = 0.7) -> dict:
    """
    POST /chat with body {message, temperature}.
    - 200 -> return the parsed JSON ({reply, model})
    - non-200 -> {'error': ..., 'status': code}
    - connection failure -> {'error': ...}
    """
    # TODO: try:
    #     resp = client.post('/chat', json={'message': message, 'temperature': temperature})
    # TODO: except Exception as e:
    #     return {'error': f'request failed: {e}'}
    # TODO: if resp.status_code != 200:
    #     return {'error': f'backend returned {resp.status_code}', 'status': resp.status_code}
    # TODO: return resp.json()
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    backend = TestClient(build_api())

    # Check 1: valid call returns reply + model (Ollama)
    try:
        out = post_chat(backend, 'Reply with exactly: pong')
        assert isinstance(out, dict), f'expected dict, got {type(out).__name__}'
        assert 'reply' in out and 'model' in out, f'missing contract keys: {out}'
        passed += 1; print('✅ Check 1: valid call -> {reply, model}')
    except Exception as e:
        print(f'❌ Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: reply is a non-empty string
    try:
        out = post_chat(backend, 'Say hello in three words.')
        assert isinstance(out['reply'], str) and len(out['reply']) > 0
        passed += 1; print(f"✅ Check 2: reply is non-empty ({len(out['reply'])} chars)")
    except Exception as e:
        print(f'❌ Check 2: {e}')

    # Check 3: empty message violates the contract -> error with status 422
    try:
        out = post_chat(backend, '')
        assert 'error' in out, f'expected an error dict, got {out}'
        assert out.get('status') == 422, f"expected 422, got {out.get('status')}"
        passed += 1; print('✅ Check 3: empty message -> error (status 422)')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: connection failure -> error dict (no crash)
    try:
        class _Dead:
            def post(self, *a, **k):
                raise httpx.ConnectError('refused')
        out = post_chat(_Dead(), 'hi')
        assert 'error' in out, 'connection failure must return an error dict'
        passed += 1; print('✅ Check 4: backend down -> error dict')
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: the client sends the message under the agreed key
    try:
        spy = FastAPI()
        @spy.post('/chat')
        def _echo(req: ChatRequest):
            return {'reply': req.message, 'model': 'echo'}
        out = post_chat(TestClient(spy), 'contract-check')
        assert out.get('reply') == 'contract-check', f'contract mismatch: {out}'
        passed += 1; print('✅ Check 5: JSON contract keys line up front-to-back')
    except Exception as e:
        print(f'❌ Check 5: {e}')

    if passed == total:
        print('🎉 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
def post_chat(client, message: str, temperature: float = 0.7) -> dict:
    """POST /chat honouring the JSON contract {message, temperature}.

    Returns the parsed {reply, model} on 200. On a non-200 status returns
    {'error': ..., 'status': code}; on a connection failure returns
    {'error': ...}. The frontend never sees a raw exception.
    """
    try:
        resp = client.post('/chat', json={'message': message, 'temperature': temperature})
    except Exception as e:
        return {'error': f'request failed: {e}'}
    if resp.status_code != 200:
        return {'error': f'backend returned {resp.status_code}', 'status': resp.status_code}
    return resp.json()
```

**Why this works:** The `json=` argument serialises the dict to a JSON body with the keys `message` and `temperature` — the exact fields the backend's `ChatRequest` declares. That agreement *is* the contract; the spy backend in Check 5 proves the keys line up. Non-200 responses become an `{'error', 'status'}` dict so the UI can show 'backend returned 422' instead of throwing. The connection-error branch keeps the frontend alive when the backend isn't running yet.
</details>